# Leer diccionarios y juntar dataframes

In [1]:
import pandas as pd
import glob
import os

In [2]:
# Ruta relativa desde notebooks/
ruta_diccionarios = "../data/external/terminos/Diccionarios_de_literatura/"

# Obtener todos los archivos Excel
archivos = glob.glob(os.path.join(ruta_diccionarios, "*.xlsx"))
# 
# Lista para almacenar dataframes
dfs = []

In [3]:
archivos

['../data/external/terminos/Diccionarios_de_literatura\\Literatura latinoamericana reporte_P_v1.xlsx',
 '../data/external/terminos/Diccionarios_de_literatura\\Literatura_Colombiana_P_v1.xlsx',
 '../data/external/terminos/Diccionarios_de_literatura\\Literatura_Infantil_P_v1.xlsx',
 '../data/external/terminos/Diccionarios_de_literatura\\Literatura_juvenil_P_v1.xlsx',
 '../data/external/terminos/Diccionarios_de_literatura\\Literatura_universal_reporte_P_v2.xlsx']

In [4]:
for archivo in archivos:
    # Leer archivo
    df_temp = pd.read_excel(archivo)
    
    # Extraer nombre del archivo sin ruta ni extensión
    nombre_archivo = os.path.splitext(os.path.basename(archivo))[0]
    
    # Añadir columna identificadora
    df_temp["fuente"] = nombre_archivo
    
    dfs.append(df_temp)

# Concatenar todo en un solo dataframe
df_total = pd.concat(dfs, ignore_index=True)

print(df_total.shape)

(45415, 10)


In [5]:
df_total.head()

,instance_primary_contributor,contributors,extraido,title,notes,publication_site,publisher,dateOfPublication,pais_origen,fuente
0,Sin valor,Sin valor,True,Hombres y engranajes,Sin datos desde el origen (excel,Sin valor,Sin valor,Sin valor,Argentina,Literatura latinoamericana reporte_P_v1
1,Sin valor,Sin valor,True,La cautiva ; el matadero,Sin datos desde el origen (excel,Sin valor,Sin valor,Sin valor,Argentina,Literatura latinoamericana reporte_P_v1
2,Sin valor,Sin valor,True,Homo atomicus,Sin datos desde el origen (excel,Sin valor,Sin valor,Sin valor,Argentina,Literatura latinoamericana reporte_P_v1
3,Sin valor,Sin valor,True,Obras completas : falsificaciones,Sin datos desde el origen (excel,Sin valor,Sin valor,Sin valor,Argentina,Literatura latinoamericana reporte_P_v1
4,Sin valor,Sin valor,True,Arengas de Bartolomé Mitre : colección de disc...,Sin datos desde el origen (excel,Sin valor,Sin valor,Sin valor,Argentina,Literatura latinoamericana reporte_P_v1


In [6]:
df_total["title_author"] = (
    df_total["title"].fillna("").str.strip() +
    " | " +
    df_total["instance_primary_contributor"].fillna("").str.strip()
)


In [7]:
df_total.nunique()

instance_primary_contributor    13128
contributors                    14231
extraido                            2
title                           30883
notes                           12094
publication_site                  691
publisher                        5161
dateOfPublication                 629
pais_origen                        18
fuente                              5
title_author                    31689
dtype: int64

In [8]:
df_total['title'].value_counts()

title
Magazin dominical                                                                                     853
Espigas eucarísticas                                                                                  481
Boletin mensual de estadistica                                                                        446
La defensa católica : periódico doctrinal órgano del consejo superior del apostolado de la oración    226
La buena prensa : periódico semanal, dedicado a defender los intereses religiosos y de la patria      203
                                                                                                     ... 
El cuadro Ma. Isabel Sánchez Vegara, ilustrado por Albert Arrayás                                       1
Ansiedades Daniel Villamizar                                                                            1
Cuando las casas regresaron flotando Einar Turkowski                                                    1
Cibeles : diosa de las montañas : mito g

Verificar cuantos nombres hay en común entre instance_primary_contributor y contributors

In [9]:
set_primary = set(df_total["instance_primary_contributor"].dropna())
set_contributors = set(df_total["contributors"].dropna())

no_estan = set_primary - set_contributors

len(no_estan)

3902

In [10]:
en_comun = set_primary & set_contributors
len(en_comun)


9226

Verificar cuantos de latinoamericana están en universal

In [11]:
df_total["fuente"].unique()


array(['Literatura latinoamericana reporte_P_v1',
       'Literatura_Colombiana_P_v1', 'Literatura_Infantil_P_v1',
       'Literatura_juvenil_P_v1', 'Literatura_universal_reporte_P_v2'],
      dtype=object)

In [12]:
# Filtrar cada subconjunto
latam = df_total[df_total["fuente"].str.contains("latinoamericana", case=False)]
universal = df_total[df_total["fuente"].str.contains("universal", case=False)]

# Crear sets únicos de títulos (sin NaN)
titulos_latam = set(latam["title"].dropna())
titulos_universal = set(universal["title"].dropna())

# Intersección
titulos_repetidos = titulos_latam & titulos_universal

len(titulos_repetidos)


55

In [13]:
# Filtrar cada subconjunto
latam = df_total[df_total["fuente"].str.contains("latinoamericana", case=False)]
universal = df_total[df_total["fuente"].str.contains("universal", case=False)]

# Crear sets únicos de títulos (sin NaN)
titulos_latam = set(latam["title_author"].dropna())
titulos_universal = set(universal["title_author"].dropna())

# Intersección
titulos_repetidos = titulos_latam & titulos_universal

len(titulos_repetidos)

4

In [14]:
titulos_repetidos

{'Antologia poetica | Ruben Dario',
 'La otra literatura latinoamericana | Juan Gustavo Cobo Borda',
 'Nueva historia de la gran literatura iberoamericana | Arturo Torres Rioseco',
 'Obras escogidas | Sin valor'}

# Sin tildes y en minúscula

In [15]:
import unicodedata
import pandas as pd

def normalizar(texto):
    if pd.isna(texto):
        return texto
    texto = str(texto).strip()
    #texto = texto.lower()
    texto = unicodedata.normalize("NFD", texto)
    texto = "".join(c for c in texto if unicodedata.category(c) != "Mn")
    return texto

# Copiar dataframe
df_sin_tildes = df_total.copy()

# Aplicar solo a columnas de texto
columnas_texto = df_sin_tildes.select_dtypes(include=["object", "string"]).columns

df_sin_tildes[columnas_texto] = df_sin_tildes[columnas_texto].applymap(normalizar)



C:\Users\karen\AppData\Local\Temp\ipykernel_19532\2321614264.py:19: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_sin_tildes[columnas_texto] = df_sin_tildes[columnas_texto].applymap(normalizar)


In [16]:
df_sin_tildes.head()


,instance_primary_contributor,contributors,extraido,title,notes,publication_site,publisher,dateOfPublication,pais_origen,fuente,title_author
0,Sin valor,Sin valor,True,Hombres y engranajes,Sin datos desde el origen (excel,Sin valor,Sin valor,Sin valor,Argentina,Literatura latinoamericana reporte_P_v1,Hombres y engranajes | Sin valor
1,Sin valor,Sin valor,True,La cautiva ; el matadero,Sin datos desde el origen (excel,Sin valor,Sin valor,Sin valor,Argentina,Literatura latinoamericana reporte_P_v1,La cautiva ; el matadero | Sin valor
2,Sin valor,Sin valor,True,Homo atomicus,Sin datos desde el origen (excel,Sin valor,Sin valor,Sin valor,Argentina,Literatura latinoamericana reporte_P_v1,Homo atomicus | Sin valor
3,Sin valor,Sin valor,True,Obras completas : falsificaciones,Sin datos desde el origen (excel,Sin valor,Sin valor,Sin valor,Argentina,Literatura latinoamericana reporte_P_v1,Obras completas : falsificaciones | Sin valor
4,Sin valor,Sin valor,True,Arengas de Bartolome Mitre : coleccion de disc...,Sin datos desde el origen (excel,Sin valor,Sin valor,Sin valor,Argentina,Literatura latinoamericana reporte_P_v1,Arengas de Bartolome Mitre : coleccion de disc...


In [17]:
df_sin_tildes.nunique()

instance_primary_contributor    13105
contributors                    14186
extraido                            2
title                           30862
notes                           12087
publication_site                  653
publisher                        4980
dateOfPublication                 629
pais_origen                        18
fuente                              5
title_author                    31673
dtype: int64

In [18]:
df_sin_tildes.to_excel("../data/processed/diccionarios_literatura_sin_tildes.xlsx", index=False)